# African Wildlife Object Detection — YOLO11n fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Estiyar/DL-Bonus/blob/main/notebooks/train_detector.ipynb)

This notebook runs the full pipeline from a **fresh Colab runtime**:

1. install pinned dependencies,
2. download the dataset, fix cross-split duplicates and wrong labels, check labels, show ground truth,
3. train the **baseline** (imgsz 640) and the **experiment** (imgsz 320 — the only change),
4. compare both runs on the **validation** set and select the final model,
5. evaluate the selected checkpoint **once** on the held-out **test** set,
6. show predictions and failure cases.

**Runtime → Change runtime type → T4 GPU** (free tier is enough). Full training of both runs takes roughly 25–35 min on a T4.
Settings in the next cell: `TRAIN = False` skips training and downloads the released checkpoint; `EPOCHS = 2` runs a quick smoke test (`None` = 40 epochs from the configs).

In [ ]:
REPO_URL = "https://github.com/Estiyar/DL-Bonus.git"
WEIGHTS_URL = "https://github.com/Estiyar/DL-Bonus/releases/download/v1.0/baseline_best.pt"
TRAIN = True
EPOCHS = None

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU - training will be slow (Runtime -> Change runtime type -> GPU)"
import os
if not os.path.exists("DL-Bonus"):
    !git clone -q {REPO_URL}
%cd DL-Bonus
!pip install -q -r requirements.txt
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
from IPython.display import Image, Markdown, display
import json
def show(path, width=1000):
    display(Image(filename=path, width=width))
def show_md(path):
    display(Markdown(open(path).read()))

## 2. Data
`prepare_data.py` downloads the archive, verifies its SHA-256, moves the near-duplicate images listed in
`configs/split_fixes.yaml` (so that every duplicate group lives in one split), corrects the 7 wrong class ids
listed in `configs/label_fixes.yaml` and writes `data.yaml`.
`check_dataset.py` counts images/objects, validates every label and searches for near-duplicates with a perceptual hash.

In [ ]:
!python scripts/prepare_data.py
!python scripts/check_dataset.py

In [ ]:
!python scripts/visualize_gt.py --split train --n 8
show("results/dataset/gt_examples_train.jpg")

## 3. Training
Both configs are identical except `imgsz` (640 vs 320): same split, same `yolo11n.pt` COCO checkpoint, seed 42,
40 epochs, batch 16, SGD lr0=0.01. `best.pt` is chosen by Ultralytics on **validation** fitness.

In [ ]:
ep = f"--epochs {EPOCHS}" if EPOCHS else ""
if TRAIN:
    !python scripts/train.py --config configs/baseline.yaml {ep}
    !python scripts/train.py --config configs/exp_imgsz320.yaml {ep}
else:
    os.makedirs("weights", exist_ok=True)
    !wget -q -O weights/baseline_best.pt {WEIGHTS_URL}
    print("Downloaded released checkpoint; comparison below uses the saved metrics in results/")

In [ ]:
show("results/baseline/results.png")
show("results/exp_imgsz320/results.png")

## 4. Controlled experiment — validation comparison and final-model selection

In [ ]:
!python scripts/compare_runs.py baseline exp_imgsz320
show_md("results/comparison.md")
show("results/learning_curves_comparison.png")
final = json.load(open("results/final_run.json"))
final

## 5. Final evaluation on the held-out test set
Evaluated **once**, after the model was selected on validation. Settings: `conf=0.001`, NMS `iou=0.7`;
a prediction is a true positive for mAP@0.50 when IoU ≥ 0.50 with a ground-truth box of the same class.

In [ ]:
!python scripts/evaluate.py --weights {final['weights']} --imgsz {final['imgsz']} --split test
m = json.load(open("results/test/metrics_test.json"))
print(f"TEST  precision={m['precision']:.3f}  recall={m['recall']:.3f}  mAP@0.50={m['mAP50']:.3f}  mAP@0.50:0.95={m['mAP50_95']:.3f}")
show("results/test/confusion_matrix.png", 600)

## 6. Predictions on test images

In [ ]:
!python scripts/predict.py --weights {final['weights']} --imgsz {final['imgsz']} --n 8
show("results/test/predictions.jpg")

## 7. Failure analysis (conf ≥ 0.25, match IoU ≥ 0.5)  — left: ground truth, right: prediction

In [ ]:
!python scripts/failure_analysis.py --weights {final['weights']} --imgsz {final['imgsz']}
import glob
for f in sorted(glob.glob("results/failures/failure_*.jpg"))[:3]:
    show(f)

## 8. Try your own image

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    !python scripts/predict.py --weights {final['weights']} --imgsz {final['imgsz']} --source "{name}" --out my_prediction.jpg
    show("my_prediction.jpg", 600)